# Leeds property listings with planning applications and constraints, mapped

Searches Rightmove and Zoopla for homes for sale in Leeds with
[UK Property Listings + Planning](https://apify.com/yappman/uk-property-listings).
Each listing comes back with planning constraints from the government's planning data service at its coordinates
(England only) and planning applications within 150 metres from the last year.
The notebook maps every listing and colours it by what was found.

**Cost at Apify Free-tier prices:** at most **$0.365** per run.

| Event | Count | Each | Most |
|---|---|---|---|
| Listing | 20 | $0.003 | $0.06 |
| Planning constraints | 20 | $0.005 | $0.10 |
| Planning applications | 20 | $0.01 | $0.20 |
| Run start | 1 | $0.005 | $0.005 |

Constraints are charged only when the lookup succeeds. Applications are not
charged for areas PlanIt does not cover.

More data: [data.yappman.com](https://data.yappman.com).

In [ ]:
import os
from decimal import Decimal

from pathlib import Path

import pandas as pd
from apify_client import ApifyClient

TOKEN = os.environ.get("APIFY_TOKEN")
if not TOKEN:
    raise RuntimeError("Set the APIFY_TOKEN environment variable first. See the README.")
client = ApifyClient(TOKEN)

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)  # git-ignored: nothing you collect is committed


def run_actor(actor_id: str, run_input: dict, max_charge_usd: float):
    """Start an Actor, wait for it, and return the finished run.

    Spend is capped twice: maxItems in run_input limits the rows saved, and
    max_total_charge_usd stops the run if its charges would pass the cap.
    """
    run = client.actor(actor_id).call(
        run_input=run_input,
        max_total_charge_usd=Decimal(str(max_charge_usd)),
    )
    if run is None or run.status != "SUCCEEDED":
        raise RuntimeError(f"{actor_id} did not succeed: {None if run is None else run.status}")
    print(f"{actor_id}: run {run.id}")
    print("Charged events:", run.charged_event_counts)
    return run

## 1. Search with planning enrichment

In [ ]:
ACTOR = "yappman/uk-property-listings"

# Free tier: 20 x ($0.003 + $0.005 + $0.01) + $0.005 = $0.365
run = run_actor(
    ACTOR,
    {
        "mode": "search",
        "portals": ["rightmove", "zoopla"],
        "locations": ["Leeds"],
        "listingType": "for-sale",
        "maxItems": 20,  # shared across both portals
        "includePlanningConstraints": True,
        "includePlanningApplications": True,
        "planningRadiusMetres": 150,
        "planningYears": 1,
    },
    max_charge_usd=0.37,
)
listings = list(client.dataset(run.default_dataset_id).iterate_items())
print(len(listings), "listings")

## 2. Flatten

A constraint counts as present when it has any value other than null or false.
Flood zone values are shown as the zone number. The same home can appear on
both portals; likely duplicates (same point, price and bedrooms) are flagged,
not removed.

In [ ]:
def present_constraints(planning: dict | None) -> dict:
    constraints = (planning or {}).get("constraints") or {}
    return {k: v for k, v in constraints.items() if v not in (None, False, "", [], {})}


records = []
for item in listings:
    planning = item.get("planning") or {}
    apps = planning.get("applications") or []
    records.append({
        "portal": item.get("portal"),
        "address": item.get("address"),
        "price": (item.get("price") or {}).get("value"),
        "beds": item.get("beds"),
        "propertyType": item.get("propertyType"),
        "latitude": item.get("latitude"),
        "longitude": item.get("longitude"),
        "url": item.get("url"),
        "constraints": present_constraints(planning),
        "applications": apps,
        "applicationCount": len(apps),
        "applicationsCoverage": (planning.get("coverage") or {}).get("applications"),
    })

df = pd.DataFrame(records)
df["constraintCount"] = df["constraints"].str.len()
df["possibleDuplicate"] = df.assign(
    lat4=df["latitude"].round(4), lon4=df["longitude"].round(4)
).duplicated(["lat4", "lon4", "price", "beds"], keep=False)

df[["portal", "address", "price", "beds", "constraintCount", "applicationCount",
    "applicationsCoverage", "possibleDuplicate"]]

In [ ]:
# Which constraints turned up, and how often
pd.Series([k for c in df["constraints"] for k in c]).value_counts().rename("listings")

In [ ]:
# Nearby planning applications, one row each
apps = pd.DataFrame([
    {"listing": r.address, **a} for r in df.itertuples() for a in r.applications
])
if not apps.empty:
    apps = apps[["listing", "reference", "authority", "state", "type", "description",
                 "receivedAt", "decidedAt", "distanceMetres", "url"]]
    apps.to_csv(OUTPUT / "leeds_planning_applications.csv", index=False)
apps.head(20) if not apps.empty else "No applications within 150 m"

## 3. Map

* **Red:** at least one planning constraint.
* **Orange:** no constraints, but planning applications nearby.
* **Blue:** neither.

The map is also saved to `output/leeds_map.html`.

In [ ]:
import html
from urllib.parse import urlparse

import folium

pts = df.dropna(subset=["latitude", "longitude"])
m = folium.Map(location=[pts["latitude"].mean(), pts["longitude"].mean()], zoom_start=12)


def e(value) -> str:
    return html.escape(str(value if value is not None else ""), quote=True)


def link(url, text) -> str:
    """A link only for http(s) URLs; anything else is shown as plain text."""
    u = str(url or "").strip()
    if urlparse(u).scheme.lower() in ("http", "https"):
        return f'<a href="{e(u)}" target="_blank" rel="noopener noreferrer">{e(text)}</a>'
    return e(text)


def popup_html(r) -> str:
    price = f"£{r.price:,.0f} · " if pd.notna(r.price) else ""
    lines = [f"<b>{e(r.address)}</b>",
             f"{price}{e(r.beds)} bed · {e(r.portal)}",
             link(r.url, "Listing")]
    if r.constraints:
        lines.append("<b>Constraints:</b> " + ", ".join(f"{e(k)} ({e(v)})" for k, v in r.constraints.items()))
    for a in r.applications[:5]:
        desc = e(str(a.get("description") or "")[:120])
        lines.append(f'{link(a.get("url"), a.get("reference"))} '
                     f'{e(a.get("state"))}, {e(a.get("distanceMetres"))} m: {desc}')
    if r.applicationCount > 5:
        lines.append(f"…and {r.applicationCount - 5} more")
    return "<br>".join(lines)


for r in pts.itertuples():
    colour = "red" if r.constraintCount else ("orange" if r.applicationCount else "blue")
    folium.CircleMarker(
        location=[r.latitude, r.longitude], radius=7, color=colour,
        fill=True, fill_opacity=0.8,
        popup=folium.Popup(popup_html(r), max_width=360),
    ).add_to(m)

m.save(OUTPUT / "leeds_map.html")
m

## Notes

* Constraints cover England only. Planning applications come from a nightly copy
  of PlanIt; check `applicationsCoverage` before reading "no applications" as
  "nothing planned".
* Listing data belongs to the Actor's data terms, not to this repository's MIT
  licence. Everything is written to `output/`, which git ignores.